# 🐘 PostgreSQL → CouchDB Migration Pipeline
**Source:** Neon PostgreSQL (cloud)  |  **Target:** Apache CouchDB

### Pipeline Steps
1. Install dependencies
2. Upload CSV → PostgreSQL
3. Extract PostgreSQL schema metadata
4. AI drafts migration plan (Azure GPT-4o) + Human review loop
5. Execute migration (PostgreSQL → CouchDB)
6. Query validation (side-by-side comparison)

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Cell 1: Install all required libraries
# psycopg2-binary  — PostgreSQL driver (no compile needed)
# sqlalchemy       — ORM / connection layer
# couchdb          — CouchDB Python client
# openai           — Azure OpenAI SDK
# pandas / tqdm    — CSV loading + progress bars
!pip install sqlalchemy psycopg2-binary couchdb openai pandas tqdm python-dotenv

# Verify psycopg2 can be imported (catches silent install failures)
import psycopg2
print(f'✅ psycopg2 version: {psycopg2.__version__}')
import couchdb
print(f'✅ couchdb module ready')


## ⚙️ Cell 2 — Configuration (Edit your credentials here)

In [ ]:
# Cell 2: Central Configuration — edit these values before running

# ── PostgreSQL / Neon (Source) ─────────────────────────────────────────
# Paste your full Neon connection string here.
# The ?sslmode=require&channel_binding=require params are kept as-is.
PG_URL = (
    "postgresql+psycopg2://neondb_owner:npg_BxcqsE5DZeH4"
    "@ep-dawn-leaf-amfv43ci-pooler.c-5.us-east-1.aws.neon.tech"
    "/neondb?sslmode=require"
)
# NOTE: channel_binding=require is a libpq option not supported in the
# SQLAlchemy URL string — we pass it via connect_args instead (see below).
PG_CONNECT_ARGS = {
    "sslmode": "require",
    "options": "-c channel_binding=require",
}

# Schema to read tables from (default 'public' for Neon)
PG_SCHEMA = "public"

# ── CouchDB (Target) ──────────────────────────────────────────────────
COUCH_HOST     = "http://localhost:5984"   # change if remote
COUCH_USER     = "admin"
COUCH_PASSWORD = "admin123"
COUCH_DB_NAME  = "migrated_db"            # logical label only; each table
                                           # gets its own CouchDB database

# ── Azure OpenAI ──────────────────────────────────────────────────────
AZURE_ENDPOINT    = "https://openai-04.openai.azure.com/"
AZURE_API_KEY     = "YOUR_AZURE_API_KEY_HERE"
AZURE_API_VERSION = "2024-12-01-preview"
DEPLOYMENT_NAME   = "gpt-4o"

# ── Quick connection smoke-test ────────────────────────────────────────
from sqlalchemy import create_engine, text

_engine = create_engine(PG_URL, connect_args=PG_CONNECT_ARGS)
try:
    with _engine.connect() as _c:
        ver = _c.execute(text('SELECT version()')).scalar()
    print(f'✅ PostgreSQL connected:\n   {ver}')
except Exception as e:
    print(f'❌ PostgreSQL connection failed: {e}')

import couchdb as _couchdb
try:
    _cs = _couchdb.Server(COUCH_HOST)
    _cs.resource.credentials = (COUCH_USER, COUCH_PASSWORD)
    list(_cs)   # triggers auth
    print(f'✅ CouchDB connected: {COUCH_HOST}')
except Exception as e:
    print(f'❌ CouchDB connection failed: {e}')


## 📂 Cell 3 — Upload CSV Files into PostgreSQL

In [ ]:
# Cell 3: Load CSV files into PostgreSQL tables.
# ─── How to use ──────────────────────────────────────────────────────
# 1. Add paths to your CSV files in CSV_FILES below.
# 2. Table name = CSV filename (lowercased, spaces → underscores).
# 3. Run this cell — creates/replaces the table automatically.
# If your data is already in PostgreSQL, skip this cell.
# ─────────────────────────────────────────────────────────────────────

import pandas as pd
from sqlalchemy import create_engine

CSV_FILES = [
    # "path/to/your_file.csv",
    # "path/to/another_file.csv",
]

pg_engine = create_engine(PG_URL, connect_args=PG_CONNECT_ARGS)

if not CSV_FILES:
    print('⚠️  No CSV files specified. Add paths to CSV_FILES above, or skip this cell.')
else:
    for csv_path in CSV_FILES:
        table_name = (
            csv_path.split('/')[-1]
            .replace('.csv', '')
            .replace(' ', '_')
            .lower()
        )
        print(f'📂 Loading "{csv_path}" → PostgreSQL table "{table_name}" ...')
        df = pd.read_csv(csv_path)

        # Normalise column names: lowercase + underscores (Postgres convention)
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

        df.to_sql(
            name=table_name,
            con=pg_engine,
            schema=PG_SCHEMA,
            if_exists='replace',   # 'append' to add rows without dropping
            index=False,
            chunksize=1000,
            method='multi',        # faster bulk inserts for Postgres
        )
        print(f'   ✅ {len(df)} rows inserted into "{PG_SCHEMA}"."{table_name}"')

    print('\n🏁 All CSV files loaded into PostgreSQL.')


## 🔍 Cell 4 — Extract PostgreSQL Schema Metadata

In [ ]:
# Cell 4: Connect to PostgreSQL and extract a rich schema summary.
# Uses psycopg2 directly against information_schema for maximum detail.

import psycopg2
import psycopg2.extras
from sqlalchemy import create_engine, inspect as sa_inspect

def extract_pg_schema(pg_url, connect_args, schema='public'):
    """
    Returns a detailed text summary of every table in `schema`:
    columns, types, nullable, PKs, FKs, incoming references.
    Uses SQLAlchemy inspector so it works with any psycopg2-compatible URL.
    """
    engine    = create_engine(pg_url, connect_args=connect_args)
    inspector = sa_inspect(engine)

    table_names = inspector.get_table_names(schema=schema)
    if not table_names:
        return f'⚠️  No tables found in schema "{schema}"."

    final_output = []

    for table_name in table_names:
        output = []
        output.append(f'\n📋 Table: {schema}.{table_name}')
        output.append('-' * 80)

        # ── Columns ────────────────────────────────────────────────
        cols = inspector.get_columns(table_name, schema=schema)
        output.append(f'{"Column":<30} {"Type":<20} {"Nullable"}')
        output.append('-' * 80)
        for col in cols:
            nullable = 'YES' if col.get('nullable', True) else 'NO'
            output.append(f'{col["name"]:<30} {str(col["type"]):<20} {nullable}')

        # ── Primary Keys ───────────────────────────────────────────
        pk = inspector.get_pk_constraint(table_name, schema=schema)
        if pk and pk.get('constrained_columns'):
            output.append(f'\n🔑 Primary Key: {", ".join(pk["constrained_columns"])}')

        # ── Foreign Keys (outgoing) ────────────────────────────────
        fks = inspector.get_foreign_keys(table_name, schema=schema)
        if fks:
            output.append('\n🔗 Foreign Keys:')
            for fk in fks:
                src_cols = ', '.join(fk['constrained_columns'])
                ref_cols = ', '.join(fk['referred_columns'])
                output.append(
                    f'   [{src_cols}] → {fk["referred_schema"] or schema}'
                    f'.{fk["referred_table"]}.{ref_cols}'
                )

        # ── Row count ─────────────────────────────────────────────
        try:
            with engine.connect() as conn:
                cnt = conn.execute(
                    text(f'SELECT COUNT(*) FROM "{schema}"."{table_name}"')
                ).scalar()
            output.append(f'\n📊 Row count: {cnt:,}')
        except Exception:
            pass

        final_output.append('\n'.join(output))

    return '\n\n'.join(final_output)


print('🔎 Extracting PostgreSQL schema ...')
schema_text = extract_pg_schema(PG_URL, PG_CONNECT_ARGS, schema=PG_SCHEMA)
print(schema_text)


## 🤖 Cells 5+6+7 — AI Plan + Human Iterative Review Loop

In [ ]:
# Cells 5+6+7: AI generates the migration plan, human reviews interactively.
# The loop continues until you type 'approve'.

import json
from sqlalchemy import create_engine, inspect as sa_inspect
from openai import AzureOpenAI

ai_client = AzureOpenAI(
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
)

# ── STEP 1: get all table names from Postgres ─────────────────────────
pg_engine    = create_engine(PG_URL, connect_args=PG_CONNECT_ARGS)
pg_inspector = sa_inspect(pg_engine)
ALL_TABLES   = pg_inspector.get_table_names(schema=PG_SCHEMA)
TOTAL_TABLES = len(ALL_TABLES)

print(f'📊 PostgreSQL schema "{PG_SCHEMA}" has {TOTAL_TABLES} table(s): {ALL_TABLES}')
print('   AI will generate a plan entry for EVERY table above.\n')

# ── STEP 2: rich schema string for AI context ─────────────────────────
def build_full_schema_text(inspector, schema, table_names):
    lines = []
    for table in table_names:
        cols    = inspector.get_columns(table, schema=schema)
        col_str = ', '.join(f"{c['name']} ({c['type']})" for c in cols)

        pk = inspector.get_pk_constraint(table, schema=schema)
        pk_str = f"  PK: {pk['constrained_columns']}" if pk.get('constrained_columns') else ''

        fks = inspector.get_foreign_keys(table, schema=schema)
        fk_str = ''
        if fks:
            parts = [
                f"{fk['constrained_columns']} → {fk['referred_table']}.{fk['referred_columns']}"
                for fk in fks
            ]
            fk_str = f'  FKs: {"; ".join(parts)}'

        lines.append(f'Table: {table}\n  Columns: {col_str}{pk_str}{fk_str}')
    return '\n\n'.join(lines)

full_schema_text = build_full_schema_text(pg_inspector, PG_SCHEMA, ALL_TABLES)
print('📋 Full schema sent to AI:')
print(full_schema_text)

# ── STEP 3: system prompt ─────────────────────────────────────────────
def build_system_prompt(all_tables):
    table_list_str = '\n'.join(f'  - {t}' for t in all_tables)
    return f"""
You are a Database Architect migrating data from PostgreSQL to Apache CouchDB.

CouchDB rules:
- Data lives in 'databases' (like Mongo collections).
- Every document must have a unique string '_id'.
- Prefer embedding child rows as nested arrays for One-to-Few relationships.
- Keep as separate databases for One-to-Many (large/unbounded) relationships.
- Use strictly lowercased CouchDB database names (no spaces, use underscores).

YOUR CRITICAL OBLIGATION:
The PostgreSQL database has EXACTLY {len(all_tables)} tables:
{table_list_str}

You MUST include ALL {len(all_tables)} tables in your output JSON.
DO NOT skip, merge, or omit any table.
If a child table is embedded inside a parent, it must STILL appear as its own
entry with \"embed\": [] (it won't be migrated separately).

Rules:
1. Rename snake_case / spaced SQL column names to camelCase JSON fields.
2. Set 'id_field' to the EXACT column name (verbatim from schema) that is the best
   unique identifier — usually the primary key.
3. 'foreign_key_column' MUST be the EXACT column name as it exists in the CHILD
   table's schema (copy verbatim). NEVER invent or camelCase it.
4. Include ALL columns in column_mapping (even unchanged ones).
5. When the human gives feedback, update the plan and return ALL {len(all_tables)} entries.

Output ONLY a valid JSON array with exactly {len(all_tables)} entries — no Markdown:
[
    {{
        \"sql_table\": \"exact_postgres_table_name\",
        \"couch_database\": \"target_db_name\",
        \"id_field\": \"column_name\",
        \"column_mapping\": {{
            \"original_column_name\": \"camelCaseName\"
        }},
        \"embed\": [
            {{
                \"table\": \"child_table_name\",
                \"foreign_key_column\": \"fk_col_in_child\",
                \"target_field\": \"fieldNameInParentDoc\"
            }}
        ]
    }}
]
"""

SYSTEM_PROMPT = build_system_prompt(ALL_TABLES)

# ── Helpers ───────────────────────────────────────────────────────────
def validate_plan_coverage(plan, all_tables):
    plan_tables = {job['sql_table'].lower() for job in plan}
    return [t for t in all_tables if t.lower() not in plan_tables]


def pretty_print_plan(plan, all_tables):
    print('\n' + '='*60)
    print(f'📋 CURRENT MIGRATION PLAN  ({len(plan)}/{len(all_tables)} tables)')
    print('='*60)
    for i, job in enumerate(plan, 1):
        embeds  = [e['table'] for e in job.get('embed', [])]
        col_map = job.get('column_mapping', {})
        dropped = [k for k, v in col_map.items() if v is None]
        renamed = {k: v for k, v in col_map.items() if v is not None}
        print(f'\n  [{i}] PostgreSQL Table : {job.get("sql_table")}')
        print(f'       CouchDB DB       : {job.get("couch_database")}')
        print(f'       ID Field         : {job.get("id_field", "id")}')
        if renamed: print(f'       Renamed          : {renamed}')
        if dropped: print(f'       Dropped          : {dropped}')
        print(f'       Embeds           : {embeds if embeds else "None (standalone)"}')
    missing = validate_plan_coverage(plan, all_tables)
    if missing:
        print(f'\n  ⚠️  WARNING — tables MISSING from plan:')
        for t in missing: print(f'       ✗ {t}')
        print('     AI will be asked to fix this automatically.')
    else:
        print(f'\n  ✅ All {len(all_tables)} tables are covered.')
    print('\n' + '='*60)


def call_ai(conversation_history):
    response = ai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=conversation_history,
        temperature=0,
    )
    content = response.choices[0].message.content.strip()
    if '```' in content:
        content = content.replace('```json', '').replace('```', '').strip()
    return content


# ── Main loop ─────────────────────────────────────────────────────────
print('\n🤖 Generating initial migration plan ...\n')

conversation_history = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {
        'role': 'user',
        'content': (
            f'Here is the complete PostgreSQL schema for ALL {TOTAL_TABLES} tables.\n'
            f'Generate a migration plan entry for EVERY table listed.\n\n'
            f'{full_schema_text}'
        )
    }
]

raw_response = call_ai(conversation_history)
try:
    current_plan = json.loads(raw_response)
except json.JSONDecodeError as e:
    print(f'❌ AI returned invalid JSON: {e}\nRaw: {raw_response}')
    raise

conversation_history.append({'role': 'assistant', 'content': raw_response})

# Auto-fix missing tables
missing = validate_plan_coverage(current_plan, ALL_TABLES)
if missing:
    print(f'⚠️  AI missed {len(missing)} table(s): {missing}. Auto-requesting fix ...')
    fix_msg = (
        f'Your plan is missing these PostgreSQL tables: {missing}.\n'
        f'Add them to the JSON plan. Return ALL {TOTAL_TABLES} entries in the array.'
    )
    conversation_history.append({'role': 'user', 'content': fix_msg})
    raw_response = call_ai(conversation_history)
    try:
        current_plan = json.loads(raw_response)
        conversation_history.append({'role': 'assistant', 'content': raw_response})
        print('✅ AI fixed the missing tables.')
    except json.JSONDecodeError:
        print('⚠️  AI still returned invalid JSON after fix attempt. Proceeding with partial plan.')

iteration     = 1
approved_plan = None

while True:
    pretty_print_plan(current_plan, ALL_TABLES)

    # Auto-fix before asking human
    missing = validate_plan_coverage(current_plan, ALL_TABLES)
    if missing:
        fix_msg = (
            f'Still missing tables: {missing}. '
            f'Add all missing tables and return the full {TOTAL_TABLES}-entry JSON array.'
        )
        conversation_history.append({'role': 'user', 'content': fix_msg})
        raw_response = call_ai(conversation_history)
        try:
            current_plan = json.loads(raw_response)
            conversation_history.append({'role': 'assistant', 'content': raw_response})
            continue
        except json.JSONDecodeError:
            pass

    print(f'\n💬 Iteration #{iteration}')
    print('   Type your change request and press Enter.')
    print('   Or type  "approve"  to accept and proceed to migration.')
    print('-' * 60)

    human_input = input('Your feedback > ').strip()

    if human_input.lower() in ['approve', 'approved', 'ok', 'yes', 'done', 'accept']:
        missing = validate_plan_coverage(current_plan, ALL_TABLES)
        if missing:
            print(f'\n⛔ Cannot approve — plan still missing tables: {missing}')
            print('   Type a request to fix them first.')
            continue
        approved_plan = current_plan
        with open('migration_blueprint.json', 'w') as f:
            json.dump(approved_plan, f, indent=4)
        print('\n' + '='*60)
        print('✅ PLAN APPROVED!')
        print(f'   {len(approved_plan)} table(s) queued for migration.')
        print('   Saved → migration_blueprint.json')
        print('   ➡️  Now run Cell 8 to start migration.')
        print('='*60)
        break

    if not human_input:
        print('⚠️  No input. Type a change request or "approve".')
        continue

    print('\n🤖 Updating plan ...')
    feedback_message = (
        f'Human feedback: "{human_input}"\n\n'
        f'Apply this change to the plan. '
        f'Return ALL {TOTAL_TABLES} table entries in the JSON array — '
        f'do not drop any table from the output.'
    )
    conversation_history.append({'role': 'user', 'content': feedback_message})
    raw_response = call_ai(conversation_history)
    try:
        current_plan = json.loads(raw_response)
        conversation_history.append({'role': 'assistant', 'content': raw_response})
        iteration += 1
        print('✅ Plan updated!')
    except json.JSONDecodeError as e:
        print(f'⚠️  AI returned invalid JSON: {e}')
        print('    Previous plan still active. Try rephrasing.')
        conversation_history.pop()


## 🚀 Cell 8 — Execute Migration (PostgreSQL → CouchDB)

In [ ]:
# Cell 8: Robust ETL engine.
# Reads PostgreSQL rows, validates the plan, transforms types,
# writes to CouchDB with FK auto-correction and per-row error recovery.

import decimal
import datetime
import uuid
import json
import couchdb
from sqlalchemy import create_engine, text, inspect as sa_inspect
from tqdm import tqdm


# ── Type serialiser (Postgres-aware) ─────────────────────────────────
def safe_serialize_pg(row_dict):
    """
    Converts PostgreSQL-specific Python types to plain JSON-safe types.
    Handles: Decimal, UUID, date/datetime/time, bytes, memoryview,
    psycopg2 Range types, lists/tuples (Postgres arrays).
    """
    import ipaddress
    clean = {}
    for key, value in row_dict.items():
        if value is None:
            clean[key] = None
        elif isinstance(value, decimal.Decimal):
            clean[key] = float(value)
        elif isinstance(value, uuid.UUID):
            clean[key] = str(value)
        elif isinstance(value, datetime.datetime):
            clean[key] = value.isoformat()
        elif isinstance(value, datetime.date):
            clean[key] = value.isoformat()
        elif isinstance(value, datetime.time):
            clean[key] = value.strftime('%H:%M:%S')
        elif isinstance(value, datetime.timedelta):
            clean[key] = str(value)
        elif isinstance(value, (bytes, memoryview)):
            try:
                clean[key] = bytes(value).decode('utf-8')
            except Exception:
                clean[key] = '<binary_data_skipped>'
        elif isinstance(value, (list, tuple)):
            # Postgres array → JSON array (recurse each element)
            clean[key] = safe_serialize_pg({'v': list(value)})['v']
        elif hasattr(value, '_asdict'):
            # psycopg2 composite / range type
            clean[key] = str(value)
        else:
            try:
                json.dumps(value)   # test JSON-serialisability
                clean[key] = value
            except (TypeError, ValueError):
                clean[key] = str(value)
    return clean


# ── Column mapping (case-insensitive) ────────────────────────────────
def apply_column_mapping(doc, column_mapping):
    if not column_mapping:
        return doc
    lower_map = {k.lower(): (k, v) for k, v in column_mapping.items()}
    new_doc = {}
    for k, v in doc.items():
        lookup = lower_map.get(k.lower())
        if lookup is not None:
            _, mapped_name = lookup
            if mapped_name is None:
                continue  # DROP this column
            new_doc[mapped_name] = v
        else:
            new_doc[k] = v
    return new_doc


# ── CouchDB helpers ───────────────────────────────────────────────────
def get_or_create_couch_db(couch_server, db_name):
    if db_name in couch_server:
        return couch_server[db_name]
    return couch_server.create(db_name)


def safe_bulk_save(target_db, batch):
    """Bulk save with per-document fallback on partial failures."""
    try:
        results = target_db.update(batch)
        errors  = [r for r in results if not r[0]]
        if not errors:
            return len(batch), 0
        ok = len(batch) - len(errors)
        err = 0
        for success, doc_id, info in results:
            if not success:
                for doc in batch:
                    if doc.get('_id') == doc_id:
                        try:
                            target_db.save(doc)
                            ok += 1
                        except Exception as e:
                            print(f'        ⚠️  Doc "{doc_id}" skipped: {e}')
                            err += 1
                        break
        return ok, err
    except Exception as bulk_err:
        print(f'      ⚠️  Bulk save failed ({bulk_err}), falling back to per-doc saves ...')
        ok = err = 0
        for doc in batch:
            try:
                target_db.save(doc)
                ok += 1
            except Exception as e:
                print(f'        ⚠️  Doc "{doc.get("_id")}" skipped: {e}')
                err += 1
        return ok, err


# ── Get real columns for a Postgres table ────────────────────────────
def get_pg_table_columns(inspector, table_name, schema):
    try:
        cols = inspector.get_columns(table_name, schema=schema)
        return {c['name'] for c in cols}
    except Exception:
        return set()


# ── FK column auto-correction ─────────────────────────────────────────
def find_real_fk_column(inspector, child_table, planned_fk, parent_table, schema):
    """
    Resolves the real FK column name if the AI plan got it wrong.
    Tries: exact → case-insensitive → FK constraint → heuristic.
    """
    actual_cols = get_pg_table_columns(inspector, child_table, schema)
    if not actual_cols:
        return None

    # 1. Exact
    if planned_fk in actual_cols:
        return planned_fk

    # 2. Case-insensitive
    for col in actual_cols:
        if col.lower() == planned_fk.lower():
            print(f'      🔧 FK case-fix: "{planned_fk}" → "{col}" in "{child_table}"')
            return col

    # 3. FK constraints
    try:
        fks = inspector.get_foreign_keys(child_table, schema=schema)
        for fk in fks:
            if fk.get('referred_table', '').lower() == parent_table.lower():
                real = fk['constrained_columns'][0]
                print(f'      🔧 FK resolved via constraint: "{real}" in "{child_table}"')
                return real
    except Exception:
        pass

    # 4. Heuristic name match
    hint = parent_table.lower().rstrip('s')
    candidates = [c for c in actual_cols if hint in c.lower()]
    if candidates:
        print(f'      🔧 FK heuristic match: "{candidates[0]}" in "{child_table}"')
        return candidates[0]

    return None


# ── Pre-flight validation ─────────────────────────────────────────────
def validate_plan_against_schema(migration_plan, existing_tables, inspector, schema):
    print('\n🔍 PRE-FLIGHT VALIDATION — checking plan against PostgreSQL schema ...')
    print('-' * 70)
    issues         = []
    corrected_plan = []

    for job in migration_plan:
        root_table  = job.get('sql_table')
        id_field    = job.get('id_field', 'id')
        col_map     = job.get('column_mapping', {})
        embed_rules = job.get('embed', [])

        # Resolve table name (case-insensitive)
        actual_table = None
        if root_table in existing_tables:
            actual_table = root_table
        else:
            for t in existing_tables:
                if t.lower() == root_table.lower():
                    actual_table = t
                    break

        if not actual_table:
            print(f'  ❌ SKIP  "{root_table}" — table not found in PostgreSQL schema "{schema}"')
            issues.append(f'Table "{root_table}" not found')
            continue

        root_cols = get_pg_table_columns(inspector, actual_table, schema)

        # Validate id_field
        real_id = None
        if id_field in root_cols:
            real_id = id_field
        else:
            for c in root_cols:
                if c.lower() == id_field.lower():
                    real_id = c
                    break
        if real_id is None:
            # Try PK
            pk = inspector.get_pk_constraint(actual_table, schema=schema)
            if pk.get('constrained_columns'):
                real_id = pk['constrained_columns'][0]
        if real_id is None and root_cols:
            real_id = sorted(root_cols)[0]

        if real_id != id_field:
            print(f'  🔧 ID fix  "{actual_table}": id_field "{id_field}" → "{real_id}"')

        # Validate column_mapping keys
        root_cols_lower = {c.lower(): c for c in root_cols}
        fixed_col_map = {}
        for planned_col, target_name in col_map.items():
            real_col = root_cols_lower.get(planned_col.lower())
            if real_col:
                fixed_col_map[real_col] = target_name
            else:
                print(f'  ⚠️   Column "{planned_col}" not in "{actual_table}" — removed from mapping')

        # Validate + auto-correct embed FK columns
        fixed_embeds = []
        for embed_rule in embed_rules:
            child_table  = embed_rule.get('table')
            fk_col       = embed_rule.get('foreign_key_column')
            target_field = embed_rule.get('target_field')

            actual_child = None
            if child_table in existing_tables:
                actual_child = child_table
            else:
                for t in existing_tables:
                    if t.lower() == child_table.lower():
                        actual_child = t
                        break

            if not actual_child:
                print(f'  ⚠️   Embed child "{child_table}" not found — embed skipped')
                continue

            real_fk = find_real_fk_column(inspector, actual_child, fk_col, actual_table, schema)
            if real_fk is None:
                print(f'  ❌  Cannot resolve FK "{fk_col}" in "{actual_child}" — embed skipped')
                issues.append(f'FK "{fk_col}" not found in "{actual_child}"')
                continue

            fixed_embeds.append({
                'table': actual_child,
                'foreign_key_column': real_fk,
                'target_field': target_field,
            })
            print(f'  ✅ Embed  "{actual_table}" ← "{actual_child}" via "{real_fk}"')

        job = dict(job)
        job.update({'sql_table': actual_table, 'id_field': real_id,
                    'column_mapping': fixed_col_map, 'embed': fixed_embeds})
        corrected_plan.append(job)
        print(f'  ✅ Table  "{actual_table}" → CouchDB "{job.get("couch_database")}" (id: "{real_id}", cols: {len(root_cols)})')

    print('-' * 70)
    if issues:
        print(f'  ⚠️  {len(issues)} issue(s) auto-corrected or skipped (see above).')
    else:
        print('  ✅ All checks passed — plan is valid.')
    print()
    return corrected_plan


# ── Main migration engine ─────────────────────────────────────────────
def execute_migration(pg_url, pg_connect_args, pg_schema,
                      couch_host, couch_user, couch_password,
                      migration_plan):

    print('\n🚀 Starting PostgreSQL → CouchDB Migration ...')

    # 1. Connect to PostgreSQL
    try:
        sql_engine      = create_engine(pg_url, connect_args=pg_connect_args)
        inspector       = sa_inspect(sql_engine)
        existing_tables = set(inspector.get_table_names(schema=pg_schema))
        print(f'✅ Connected to PostgreSQL. Tables in "{pg_schema}": {sorted(existing_tables)}')
    except Exception as e:
        print(f'❌ CRITICAL: PostgreSQL connection failed — {e}')
        return

    # 2. Connect to CouchDB
    try:
        couch_server = couchdb.Server(couch_host)
        couch_server.resource.credentials = (couch_user, couch_password)
        list(couch_server)
        print(f'✅ Connected to CouchDB at {couch_host}')
    except Exception as e:
        print(f'❌ CRITICAL: CouchDB connection failed — {e}')
        return

    # 3. Pre-flight validation
    with sql_engine.connect() as conn:
        corrected_plan = validate_plan_against_schema(
            migration_plan, existing_tables, inspector, pg_schema
        )

        if not corrected_plan:
            print('❌ No valid tables to migrate after validation. Aborting.')
            return

        migration_summary = []

        # 4. Migrate each table
        for job in corrected_plan:
            actual_table  = job['sql_table']
            couch_target  = job.get('couch_database', actual_table)
            id_field      = job['id_field']
            col_map       = job.get('column_mapping', {})
            embed_rules   = job.get('embed', [])

            # Skip tables that are only embedded (no standalone migration)
            if not job.get('migrate_standalone', True) and not embed_rules:
                # If embed is empty AND flagged as child-only, skip
                pass

            print(f'\n🔄 "{actual_table}" → CouchDB "{couch_target}"')

            # Clear + recreate target CouchDB db
            if couch_target in couch_server:
                couch_server.delete(couch_target)
                print(f'   🗑️   Cleared → "{couch_target}"')
            target_db = couch_server.create(couch_target)

            # Count rows
            try:
                mysql_count = conn.execute(
                    text(f'SELECT COUNT(*) FROM "{pg_schema}"."{actual_table}"')
                ).scalar()
            except Exception:
                mysql_count = None

            try:
                result    = conn.execute(
                    text(f'SELECT * FROM "{pg_schema}"."{actual_table}"')
                )
                batch     = []
                total_ok  = 0
                total_err = 0

                for row in tqdm(result, desc=f'  {actual_table}', unit='rows',
                                total=mysql_count):
                    row_dict = dict(row._mapping) if hasattr(row, '_mapping') else dict(row)
                    doc      = safe_serialize_pg(row_dict)

                    # Set _id BEFORE column mapping
                    raw_id = None
                    for candidate in [id_field, id_field.lower(), 'id', 'ID']:
                        if candidate in doc:
                            raw_id = doc[candidate]
                            break
                    doc['_id'] = str(raw_id) if raw_id is not None else str(uuid.uuid4())

                    # Apply column mapping
                    doc = apply_column_mapping(doc, col_map)

                    # Embed child tables
                    for embed_rule in embed_rules:
                        child_table  = embed_rule['table']
                        fk_col       = embed_rule['foreign_key_column']
                        target_field = embed_rule['target_field']
                        try:
                            child_q    = text(
                                f'SELECT * FROM "{pg_schema}"."{child_table}"'
                                f' WHERE "{fk_col}" = :rid'
                            )
                            child_rows = conn.execute(child_q, {'rid': raw_id})
                            children   = []
                            for crow in child_rows:
                                crow_dict = dict(crow._mapping) if hasattr(crow, '_mapping') else dict(crow)
                                children.append(safe_serialize_pg(crow_dict))
                            doc[target_field] = children
                        except Exception as embed_err:
                            print(f'    ⚠️  Embed "{child_table}" failed: {embed_err}')
                            doc[target_field] = []

                    batch.append(doc)

                    if len(batch) >= 500:
                        ok, err    = safe_bulk_save(target_db, batch)
                        total_ok  += ok
                        total_err += err
                        batch      = []

                if batch:
                    ok, err    = safe_bulk_save(target_db, batch)
                    total_ok  += ok
                    total_err += err

                couch_count = len(target_db)
                status = '✅' if (mysql_count is None or couch_count == mysql_count) else '⚠️ '
                print(
                    f'   {status} PostgreSQL: {mysql_count} rows  |  CouchDB: {couch_count} docs'
                    + (f'  |  Errors: {total_err}' if total_err else '')
                )
                migration_summary.append({
                    'table': actual_table, 'pg_rows': mysql_count,
                    'couch_docs': couch_count, 'errors': total_err
                })

            except Exception as table_err:
                print(f'❌ ERROR migrating "{actual_table}": {table_err}')

    # 5. Final summary
    print('\n' + '='*60)
    print('  📊 MIGRATION SUMMARY')
    print('='*60)
    all_ok = True
    for s in migration_summary:
        match  = s['pg_rows'] == s['couch_docs']
        icon   = '✅' if match else '⚠️ '
        all_ok = all_ok and match
        print(
            f'  {icon} {s["table"]:<30} PG={s["pg_rows"]}  CouchDB={s["couch_docs"]}'
            + (f'  Errors={s["errors"]}' if s['errors'] else '')
        )
    print('='*60)
    print('  ' + ('✅ ALL TABLES MIGRATED SUCCESSFULLY'
                  if all_ok else '⚠️  SOME COUNTS MISMATCH — review above'))
    print('='*60)
    print('\n🏁 Migration Complete.')


# ── Run ───────────────────────────────────────────────────────────────
execute_migration(
    pg_url         = PG_URL,
    pg_connect_args= PG_CONNECT_ARGS,
    pg_schema      = PG_SCHEMA,
    couch_host     = COUCH_HOST,
    couch_user     = COUCH_USER,
    couch_password = COUCH_PASSWORD,
    migration_plan = approved_plan,
)


## 🕵️ Cell 9 — Schema-Aware Query Validator (Side-by-Side Comparison)

In [ ]:
# Cell 9: AI-powered validator.
# Ask a natural language question → AI writes a PostgreSQL query + CouchDB Mango
# query → both run → results compared side-by-side.

import json
import decimal
import datetime
import couchdb
from sqlalchemy import create_engine, text, inspect as sa_inspect
from openai import AzureOpenAI


class PGToCouchValidator:

    def __init__(self):
        self.ai_client = AzureOpenAI(
            api_key=AZURE_API_KEY,
            api_version=AZURE_API_VERSION,
            azure_endpoint=AZURE_ENDPOINT,
        )

        print('🔌 Connecting to PostgreSQL ...')
        self.sql_engine = create_engine(PG_URL, connect_args=PG_CONNECT_ARGS)
        self.pg_schema  = PG_SCHEMA

        print('🔌 Connecting to CouchDB ...')
        self.couch_server = couchdb.Server(COUCH_HOST)
        self.couch_server.resource.credentials = (COUCH_USER, COUCH_PASSWORD)

        print('✅ Connected to both databases.\n')

        self.sql_schema   = self._extract_sql_schema()
        self.couch_schema = self._extract_couch_schema()

    def _extract_sql_schema(self):
        print('📊 Extracting PostgreSQL schema ...')
        inspector   = sa_inspect(self.sql_engine)
        schema_dict = {}
        try:
            for table in inspector.get_table_names(schema=self.pg_schema):
                cols = [
                    f"{c['name']} ({c['type']})"
                    for c in inspector.get_columns(table, schema=self.pg_schema)
                ]
                schema_dict[table] = cols
            return json.dumps(schema_dict, indent=2)
        except Exception as e:
            print(f'⚠️  SQL schema extraction failed: {e}')
            return '{}'

    def _extract_couch_schema(self):
        print('📊 Extracting CouchDB schema (sample docs) ...')
        schema_dict = {}
        try:
            for db_name in self.couch_server:
                db     = self.couch_server[db_name]
                sample = None
                for doc_id in db:
                    raw = dict(db[doc_id])
                    for k, v in list(raw.items()):
                        if isinstance(v, (datetime.date, datetime.datetime)):
                            raw[k] = v.isoformat()
                    sample = raw
                    break
                schema_dict[db_name] = sample if sample else 'Empty database'
            return json.dumps(schema_dict, indent=2)
        except Exception as e:
            print(f'⚠️  CouchDB schema extraction failed: {e}')
            return '{}'

    def translate_nl_to_queries(self, natural_language_query):
        print(f"\n🧠 Translating: '{natural_language_query}' ...")
        system_prompt = f"""
You are a Database Architect validating a PostgreSQL → CouchDB migration.
I will give you a natural language question.
Write a PostgreSQL SQL query AND a CouchDB Mango query that both answer the question.

CRITICAL RULES:
1. Use [SQL SCHEMA] for exact PostgreSQL column names. Quote identifiers: "schema"."table".
2. Use [COUCH SCHEMA] for exact CouchDB field names.
3. CouchDB Mango queries use the selector/fields/limit/sort format.

[SQL SCHEMA]
{self.sql_schema}

[COUCH SCHEMA (sample documents showing structure)]
{self.couch_schema}

OUTPUT FORMAT — output ONLY this strict JSON (no Markdown, no extra text):
{{
    "sql_query": "SELECT ...",
    "couch_database": "database_name_in_couchdb",
    "couch_mango": {{
        "selector": {{}},
        "fields": [],
        "limit": 25
    }}
}}
"""
        response = self.ai_client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': natural_language_query},
            ],
            response_format={'type': 'json_object'},
            temperature=0,
        )
        return json.loads(response.choices[0].message.content)

    def _serialize_row(self, row_dict):
        clean = {}
        for k, v in row_dict.items():
            if isinstance(v, decimal.Decimal):                        clean[k] = float(v)
            elif isinstance(v, datetime.datetime):                   clean[k] = v.isoformat()
            elif isinstance(v, datetime.date):                       clean[k] = v.isoformat()
            elif isinstance(v, datetime.time):                       clean[k] = str(v)
            elif isinstance(v, datetime.timedelta):                  clean[k] = str(v)
            elif isinstance(v, (bytes, memoryview)):                 clean[k] = '<binary>'
            elif hasattr(v, '_asdict'):                              clean[k] = str(v)
            else:                                                     clean[k] = v
        return clean

    def execute_and_compare(self, nl_query):
        try:
            queries       = self.translate_nl_to_queries(nl_query)
            sql_query     = queries.get('sql_query')
            couch_db_name = queries.get('couch_database')
            couch_mango   = queries.get('couch_mango')

            print('-' * 70)
            print(f'🐘 PostgreSQL Query : {sql_query}')
            print(f'🛋️  CouchDB Mango   : db={couch_db_name}, {json.dumps(couch_mango)}')
            print('-' * 70)

            # Execute PostgreSQL
            sql_results = []
            with self.sql_engine.connect() as conn:
                result = conn.execute(text(sql_query))
                for row in result:
                    row_dict = dict(row._mapping) if hasattr(row, '_mapping') else dict(row)
                    sql_results.append(self._serialize_row(row_dict))

            # Execute CouchDB Mango
            couch_results = []
            if couch_db_name in self.couch_server:
                db     = self.couch_server[couch_db_name]
                cursor = db.find(couch_mango)
                for doc in cursor:
                    d = dict(doc)
                    d.pop('_rev', None)
                    couch_results.append(d)
            else:
                print(f"⚠️  CouchDB database '{couch_db_name}' not found.")

            # Comparison
            print('\n📊 --- VALIDATION RESULTS ---')
            print(f'🐘 PostgreSQL returned : {len(sql_results)} records')
            print(f'🛋️  CouchDB returned   : {len(couch_results)} records\n')

            if sql_results:
                print('🐘 PostgreSQL Sample (first 2):')
                print(json.dumps(sql_results[:2], indent=2, default=str))

            if couch_results:
                print('\n🛋️  CouchDB Sample (first 2):')
                print(json.dumps(couch_results[:2], indent=2, default=str))

            if len(sql_results) == len(couch_results):
                print('\n✅ VALIDATION PASSED: Record counts match!')
            else:
                print('\n⚠️  VALIDATION MISMATCH: Record counts differ.')

        except Exception as e:
            print(f'\n❌ ERROR during validation: {e}')


print('='*60)
print(' 🕵️  POSTGRESQL ↔ COUCHDB MIGRATION VALIDATOR')
print('='*60)
validator = PGToCouchValidator()


## 💬 Cell 10 — Ask Questions & Validate (Run as Many Times as You Like)

In [ ]:
# Cell 10: Ask any natural language question.
# The AI writes PostgreSQL + CouchDB queries, runs both, and compares results.
# Change the question below and re-run this cell as needed.

question = "Show me the first 10 records from any table"

validator.execute_and_compare(question)


---
## 📝 Quick Reference

| Step | Cell | What it does |
|------|------|--------------|
| Install | 1 | `pip install` psycopg2-binary, couchdb, openai, pandas, tqdm |
| Config | 2 | Set Neon PostgreSQL URL + CouchDB + Azure credentials |
| Load CSV | 3 | Upload CSVs into PostgreSQL (skip if data already exists) |
| Schema | 4 | Extract PostgreSQL schema (columns, PKs, FKs, row counts) |
| AI Plan + Review | 5+6+7 | GPT-4o drafts plan, human reviews in loop until approved |
| Migrate | 8 | ETL: PostgreSQL rows → CouchDB documents (with validation) |
| Validator | 9 | Initialise the side-by-side query validator |
| Query | 10 | Ask NL questions, compare PostgreSQL vs CouchDB results |